# PathSnap: Exploratory Data Analysis (EDA) & Spatial Preprocessing
**PES University | Department of Computer Science and Engineering**  
**Course:** UE24CS352A Machine Learning Hackathon  
**Role Owner:** Member 1 (Data Pipeline, Spatial Engine & Feature Engineering)

---
### 1. Challenge & Dataset Architecture Overview
PathSnap reconstructs realistic vehicle trajectories from noisy GPS pings (nominally 1 Hz) by matching observations onto a directed road network graph. This notebook conducts comprehensive exploratory analysis across the 80 training trips and 10 student-test trips.

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.titlesize'] = 14

# Load summary statistics
with open('../data/processed/eda_summary.json', 'r') as f:
    eda_summary = json.load(f)

print(f"Total Train Trips: {eda_summary['train_trip_count']}")
print(f"Total Train GPS Observations: {eda_summary['train_total_points']:,}")
print(f"Total Test GPS Observations: {eda_summary['test_total_points']:,}")
print(f"Nominal 1 Hz Percentage: {eda_summary['sampling_1s_pct']:.2f}%")

### 2. Dataset Structure & Partition Verification
Each trip folder contains four files:
- `.track`: GPS pings `(longitude, latitude, timestamp_seconds)`
- `.nodes`: Road network vertices `(longitude, latitude)` (0-based line index is `node_id`)
- `.arcs`: Directed edges `(from_node_id, to_node_id)` (0-based line index is `arc_id`)
- `.route`: Chronological sequence of ground-truth `arc_id`s traversed by the vehicle

In [2]:
# Inspect sample trip 00000001
sample_tid = '00000001'
base_p = f'../Student folder/train/{sample_tid}/{sample_tid}'
track_df = pd.read_csv(f'{base_p}.track', sep=r'\s+', header=None, names=['lon', 'lat', 'time'])
nodes_df = pd.read_csv(f'{base_p}.nodes', sep=r'\s+', header=None, names=['lon', 'lat'])
arcs_df = pd.read_csv(f'{base_p}.arcs', sep=r'\s+', header=None, names=['from_node', 'to_node'])
route_df = pd.read_csv(f'{base_p}.route', sep=r'\s+', header=None, names=['arc_id'])

print(f"Trip {sample_tid}:")
print(f"  GPS Track rows: {len(track_df):,}")
print(f"  Road Network Nodes: {len(nodes_df):,}")
print(f"  Road Network Arcs: {len(arcs_df):,}")
print(f"  Ground Truth Route segments: {len(route_df):,}")
track_df.head()

### 3. Sampling Interval & Temporal Gap Analysis
The dataset is nominally sampled at 1 Hz. However, urban canyons, tunnels, and receiver signal locks produce gaps ($> 5$s). We analyze the interval distribution to design our HMM trellis segmentation policy.

In [3]:
dt = track_df['time'].diff().dropna()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(dt[dt <= 5.0], bins=30, color='#3B82F6', edgecolor='white')
ax1.set_title('Sampling Interval (dt <= 5s)', fontweight='bold')
ax1.set_xlabel('Delta Time (seconds)')
ax1.set_ylabel('Observation Count')

trips_df = pd.DataFrame(eda_summary['train_trips'])
ax2.hist(trips_df['points'], bins=25, color='#10B981', edgecolor='white')
ax2.set_title('Trajectory Lengths Across 80 Training Trips', fontweight='bold')
ax2.set_xlabel('GPS Observations per Trip')
ax2.set_ylabel('Trip Count')

plt.tight_layout()
plt.show()

### 4. Metric Coordinate Projection Rationale
**Why Euclidean distance in degrees is invalid:**  
Degrees of longitude converge towards the poles ($R \cdot \cos(\text{lat}) \cdot \Delta \text{lon}$). At latitude $49.85^\circ$N:
- $1^\circ$ Latitude $\approx 111,195$ meters
- $1^\circ$ Longitude $\approx 71,700$ meters  
Using unprojected Euclidean distance creates an anisotropic spatial distortion of $\approx 35.5\%$. We project to local Cartesian coordinates centered at trajectory centroid $(\text{lon}_0, \text{lat}_0)$:
$$x = R \cdot (\text{lon} - \text{lon}_0) \cdot \cos(\text{lat}_0) \cdot \frac{\pi}{180}$$
$$y = R \cdot (\text{lat} - \text{lat}_0) \cdot \frac{\pi}{180}$$

### 5. Candidate Miss Rate Comparison: Midpoint Baseline vs Segment Sampling
Querying road segments by midpoint alone with radius $R = 50$m produces a severe miss rate of **$\approx 30\%$** because long road segments ($> 50$m, up to $1,100$m) have midpoints far from their ends.  
Our engineered **Segment-Sampled KDTree** samples segments at $\le 25$m intervals, reducing the candidate miss rate to **$0.0\%$**!

In [4]:
with open('../data/processed/candidate_miss_rate_comparison.json', 'r') as f:
    miss_data = json.load(f)
df_miss = pd.DataFrame(miss_data)
display(df_miss)

### 6. Processed Feature Distributions (Contract 1)
Inspecting the engineered candidate feature matrix saved in `data/processed/train_features.parquet`.

In [5]:
train_feat = pd.read_parquet('../data/processed/train_features.parquet')
print(f"Total Candidate Feature Rows: {len(train_feat):,}")
print("Class Distribution (is_match):")
print(train_feat['is_match'].value_counts(normalize=True))

# Feature distributions by class
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=train_feat.sample(min(20000, len(train_feat))), x='is_match', y='dist_to_arc', ax=axes[0], palette=['#EF4444', '#10B981'])
axes[0].set_title('Perpendicular Distance (meters)', fontweight='bold')
axes[0].set_ylim([0, 55])

sns.boxplot(data=train_feat.sample(min(20000, len(train_feat))), x='is_match', y='heading_diff', ax=axes[1], palette=['#EF4444', '#10B981'])
axes[1].set_title('Heading Difference (degrees)', fontweight='bold')

sns.boxplot(data=train_feat.sample(min(20000, len(train_feat))), x='is_match', y='projection_ratio', ax=axes[2], palette=['#EF4444', '#10B981'])
axes[2].set_title('Projection Ratio along Arc [0, 1]', fontweight='bold')

plt.tight_layout()
plt.show()